# Mean-field critical exponents and dropout

How does dropout change the propagation of correlations through a deep network? We solve the mean-field equations for smooth and kinked activations, then measure the critical exponents and scaling collapse. Figures are saved to `runs/mean_field/`.

We treat two activation classes:

- **Smooth:** $\phi(x)=\tanh(x)$  
- **Kinked:** $\phi(x)=\max(0,x)$ (ReLU), which admits a closed-form correlation map

We model **inverted dropout** with keep probability $\rho$ applied to activations independently across neurons and **independently across inputs**:
$$
y \;\mapsto\; \frac{p}{\rho}\,y,\qquad p\sim\mathrm{Bernoulli}(\rho).
$$

For a wide FC network with weight variance $\sigma_w^2$ and bias variance $\sigma_b^2$, the mean-field recursions for a single-input variance $q$ and two-input covariance $q_{12}$ are

$$
q_{l+1} \;=\; \frac{\sigma_w^2}{\rho}\,\mathbb{E}\big[\phi(\sqrt{q_l}\,z)^2\big] \;+\; \sigma_b^2,
$$

$$
q^{12}_{l+1} \;=\; \sigma_w^2\,\mathbb{E}\big[\phi(u_1)\phi(u_2)\big] \;+\; \sigma_b^2,
$$

where $(u_1,u_2)$ is a centered Gaussian pair with
$$
\mathbb{E}[u_1^2]=\mathbb{E}[u_2^2]=q_l,\qquad \mathbb{E}[u_1 u_2]=q_l\,c_l,\qquad c_l := \frac{q^{12}_l}{q_l}.
$$

At a (dropout-dependent) variance fixed point $q_*$ we obtain the **exact** 1D correlation map
$$
F_\rho(c)\;=\;\frac{\sigma_w^2\,C(q_*,c)+\sigma_b^2}{q_*},
\qquad
C(q,c):=\mathbb{E}\big[\phi(u_1)\phi(u_2)\big].
$$

From this map we compute:

- Correlation length: $\;\xi = -1/\log\lambda$ with $\lambda = F_\rho'(c_*)$  
- Fixed-point order parameter: $m_* = 1-c_*$  
- Critical relaxation: $m_l \sim l^{-p}$ at $\chi:=F'(1)=1$  
- Dropout perturbation scaling: $m_* \sim h^{1/\delta}$ and $\xi \sim h^{-\nu_\rho}$ with $h:=1-F_\rho(1)$ (the **actual** shift induced by dropout)

We estimate the exponents with log–log fits.


## How $c_*$ and $m_*$ are computed

Throughout the notebook we use the **order parameter**
$$
m_* \;:=\; 1 - c_*,
$$
where $c_*$ is the **correlation fixed point** of the (dropout-deformed) mean-field correlation map.

For a given activation $\phi$, keep probability $\rho$, and parameters $(\sigma_w^2,\sigma_b^2)$:

1. **Variance fixed point** (dropout):
$$
\bar q_* \;=\; \frac{\sigma_w^2}{\rho}\,\mathbb E\big[\phi(\sqrt{\bar q_*}\,z)^2\big] + \sigma_b^2,
\qquad z\sim\mathcal N(0,1).
$$

2. **Correlation map** for two inputs (equal-variance case):
$$
\bar F_\rho(c)
\;=\;
\frac{\sigma_w^2\,\mathbb E\big[\phi(u_1)\phi(u_2)\big] + \sigma_b^2}{\bar q_*},
$$
where $(u_1,u_2)$ is a correlated Gaussian pair with
$$
\mathrm{Var}(u_1)=\mathrm{Var}(u_2)=\bar q_*,
\qquad
\mathrm{Corr}(u_1,u_2)=c.
$$

3. **Correlation fixed point**:
$$
c_* \text{ solves } c_* = \bar F_\rho(c_*).
$$

4. **Correlation length** is extracted from the local slope at the fixed point:
$$
\lambda \;:=\; \bar F_\rho'(c_*),
\qquad
\xi \;:=\; -\frac{1}{\ln \lambda}.
$$

Numerically we obtain $c_*$ by root finding on $g(c)=\bar F_\rho(c)-c$, and then set $m_*=1-c_*$.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "utils").is_dir():
    raise FileNotFoundError(
        "Open this notebook from the repository root or notebooks/."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / "results"
RUN_DIR = ROOT / "runs" / "mean_field"
RUN_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from scipy.optimize import brentq
import warnings

warnings.filterwarnings("ignore")

# Figure style
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "legend.fontsize": 9,
        "figure.dpi": 150,
        "text.usetex": False,
        "mathtext.fontset": "cm",
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)

# Colors (consistent across the notebook)
COLOR_SMOOTH = "#3A5F3A"  # dark green
COLOR_KINK = "#C9A961"  # gilt
COLOR_FIT = "#5C7A5C"  # lighter green
COLOR_THEORY = "#D4AF6A"  # warm gold
COLOR_GRID = "#E8E0D0"  # cream


def style_log_axes(ax, legend_loc=None):
    """Standard grid and legend styling for log plots."""
    ax.grid(
        True, which="major", alpha=0.30, linestyle="-", linewidth=0.8, color=COLOR_GRID
    )
    ax.grid(
        True, which="minor", alpha=0.15, linestyle=":", linewidth=0.5, color=COLOR_GRID
    )
    ax.set_axisbelow(True)
    if legend_loc is not None:
        ax.legend(
            loc=legend_loc, frameon=True, fancybox=True, shadow=True, framealpha=0.95
        )


# Gauss–Hermite quadrature setup
# E_{z~N(0,1)}[f(z)] = sum_i w_i f(sqrt(2) x_i)/sqrt(pi)
GH_N = 60
_x, _w = hermgauss(GH_N)
_z = np.sqrt(2.0) * _x
_w1 = _w / np.sqrt(np.pi)

# For 2D expectations with independent z1,z2 ~ N(0,1):
# E[f(z1,z2)] = sum_{i,j} (w_i w_j / pi) f(sqrt(2)x_i, sqrt(2)x_j)
_Z1 = _z[:, None]
_Z2 = _z[None, :]
_W2 = (_w[:, None] * _w[None, :]) / np.pi


def gh_expect_1d(fvals: np.ndarray) -> float:
    """Given f(z_i) on GH nodes z_i, return E[f(z)] for z~N(0,1)."""
    return float(np.sum(_w1 * fvals))


def gh_expect_2d(fvals: np.ndarray) -> float:
    """Given f(z_i,z_j) on GH grid, return E[f(z1,z2)] for iid N(0,1)."""
    return float(np.sum(_W2 * fvals))


print(f"Using GH_N = {GH_N} (1D and 2D Gauss–Hermite quadrature)")


In [ ]:
# -----------------------------
# Activations and derivatives
# -----------------------------
def phi_tanh(x):
    return np.tanh(x)


def phi_tanh_p(x):
    # d/dx tanh(x) = sech^2(x)
    c = np.cosh(x)
    return 1.0 / (c * c)


def phi_tanh_pp(x):
    # d^2/dx^2 tanh(x) = -2 tanh(x) sech^2(x)
    return -2.0 * np.tanh(x) * phi_tanh_p(x)


# -----------------------------
# Smooth (tanh) full mean field
# -----------------------------
def solve_qstar_tanh(
    sigma_w2: float,
    sigma_b2: float,
    rho: float,
    tol: float = 1e-13,
    max_iter: int = 5000,
) -> float:
    """
    Solve the dropout variance fixed point:
        q = (sigma_w2/rho) E[tanh(sqrt(q) z)^2] + sigma_b2
    via damped fixed point iteration (robust for tanh).
    """
    q = sigma_b2 + 1.0
    for _ in range(max_iter):
        sq = np.sqrt(max(q, 0.0))
        Ef2 = gh_expect_1d(phi_tanh(sq * _z) ** 2)
        q_new = (sigma_w2 / rho) * Ef2 + sigma_b2
        if abs(q_new - q) < tol * max(1.0, q_new):
            return float(q_new)
        q = 0.5 * q + 0.5 * q_new
    return float(q)


def tanh_chi_g_h(sigma_w2: float, sigma_b2: float, rho: float):
    """
    Return (q*, chi, g, h) for tanh, where:
      chi = F'(1)
      g   = F''(1)
      h   = 1 - F(1)
    """
    q = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    sq = np.sqrt(q)

    Ef2 = gh_expect_1d(phi_tanh(sq * _z) ** 2)
    Efp2 = gh_expect_1d(phi_tanh_p(sq * _z) ** 2)
    Efpp2 = gh_expect_1d(phi_tanh_pp(sq * _z) ** 2)

    chi = sigma_w2 * Efp2
    g = sigma_w2 * q * Efpp2

    F1 = (sigma_w2 * Ef2 + sigma_b2) / q
    h = 1.0 - F1
    return q, chi, g, h


def F_tanh(c: float, sigma_w2: float, sigma_b2: float, rho: float, qstar: float = None):
    """
    Full mean-field correlation map with dropout:
      F(c) = (sigma_w2 E[tanh(u1) tanh(u2)] + sigma_b2) / q*
    where (u1,u2) are correlated Gaussians with Var(u1)=Var(u2)=q* and Corr=c.
    """
    if qstar is None:
        qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1.0 + 1e-15, 1.0 - 1e-15))
    s = np.sqrt(max(0.0, 1.0 - c * c))
    sq = np.sqrt(qstar)

    u1 = sq * _Z1
    u2 = sq * (c * _Z1 + s * _Z2)
    E = gh_expect_2d(phi_tanh(u1) * phi_tanh(u2))
    return float((sigma_w2 * E + sigma_b2) / qstar)


def Fp_tanh(
    c: float, sigma_w2: float, sigma_b2: float, rho: float, qstar: float = None
):
    """
    Derivative of tanh correlation map at general c (Price theorem):
      F'(c) = sigma_w2 E[phi'(u1) phi'(u2)]
    """
    if qstar is None:
        qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1.0 + 1e-15, 1.0 - 1e-15))
    s = np.sqrt(max(0.0, 1.0 - c * c))
    sq = np.sqrt(qstar)

    u1 = sq * _Z1
    u2 = sq * (c * _Z1 + s * _Z2)
    E = gh_expect_2d(phi_tanh_p(u1) * phi_tanh_p(u2))
    return float(sigma_w2 * E)


def cstar_tanh(sigma_w2: float, sigma_b2: float, rho: float) -> float:
    """
    Solve F(c)=c for tanh, selecting the stable fixed point.

    IMPORTANT: we solve in m = 1-c (not in c directly) so we can resolve
    extremely small m when c* is very close to 1 (small dropout field).
    """
    q, chi, g, h = tanh_chi_g_h(sigma_w2, sigma_b2, rho)
    t = chi - 1.0

    # If h=0 and t<=0, the stable fixed point is exactly c*=1 (so m*=0)
    if abs(h) < 1e-15 and t <= 0.0:
        return 1.0

    def g_m(m):
        c = 1.0 - m
        return F_tanh(c, sigma_w2, sigma_b2, rho, qstar=q) - c

    # Seed from the smooth equation-of-state:
    #   h + t m - (g/2) m^2 ≈ 0  <=>  (g/2)m^2 - t m - h = 0
    disc = t * t + 2.0 * g * h
    m0 = (t + np.sqrt(max(disc, 0.0))) / max(g, 1e-16)
    m0 = float(np.clip(m0, 1e-8, 1.0))

    # Bracket around m0
    m_lo = max(1e-12, m0 / 100.0)
    m_hi = min(1.9, m0 * 100.0)
    f_lo = g_m(m_lo)
    f_hi = g_m(m_hi)

    # Expand until we get a sign change
    for _ in range(80):
        if f_lo * f_hi < 0:
            break
        m_lo = max(1e-12, m_lo / 2.0)
        m_hi = min(1.9, m_hi * 2.0)
        f_lo = g_m(m_lo)
        f_hi = g_m(m_hi)
        if m_lo <= 1e-12 and m_hi >= 1.9:
            break

    if f_lo * f_hi > 0:
        # Fallback: scan to find a bracket
        ms = np.logspace(-12, 0, 600)  # 1e-12 .. 1
        fs = np.array([g_m(m) for m in ms])
        idx = np.where(fs[:-1] * fs[1:] < 0)[0]
        if len(idx) == 0:
            # Final fallback: fixed-point iteration (slow near critical, but robust)
            c = 0.0
            for _ in range(300000):
                c_new = F_tanh(c, sigma_w2, sigma_b2, rho, qstar=q)
                c_new = float(np.clip(c_new, -0.999999999999, 0.999999999999))
                if abs(c_new - c) < 1e-14:
                    return c_new
                c = 0.5 * c + 0.5 * c_new
            return c
        a, b = ms[idx[0]], ms[idx[0] + 1]
        m_star = brentq(g_m, a, b, xtol=1e-14, rtol=1e-12, maxiter=500)
    else:
        m_star = brentq(g_m, m_lo, m_hi, xtol=1e-14, rtol=1e-12, maxiter=500)

    return 1.0 - float(m_star)


# -----------------------------
# Kinked (ReLU) exact map family
# -----------------------------
def F_relu_base(c: float) -> float:
    """Arc-cosine kernel (order 1) normalized correlation map for ReLU."""
    c = float(np.clip(c, -1.0, 1.0))
    return float((np.sqrt(max(0.0, 1.0 - c * c)) + (np.pi - np.arccos(c)) * c) / np.pi)


def F_relu(c: float, chi: float, rho: float) -> float:
    """
    ReLU correlation map with tunable slope chi and dropout keep probability rho:
      F(c) = chi * F_base(c) + 1 - chi/rho
    so that:
      F'(1) = chi
      F(1)  = 1 - chi*(1/rho - 1) = 1 - h.
    """
    return float(chi * F_relu_base(c) + 1.0 - chi / rho)


def Fp_relu(c: float, chi: float, rho: float) -> float:
    """
    Derivative:
      F_base'(c) = 1 - arccos(c)/pi
      F'(c) = chi * F_base'(c)
    """
    c = float(np.clip(c, -1.0 + 1e-15, 1.0 - 1e-15))
    return float(chi * (1.0 - np.arccos(c) / np.pi))


def cstar_relu(chi: float, rho: float) -> float:
    """
    Solve F(c)=c for ReLU, selecting the stable fixed point.

    We solve in m = 1-c with an asymptotic seed so we can resolve the tiny-m
    regime without being fooled by floating-point noise very close to c=1.
    """
    h = 1.0 - F_relu(1.0, chi, rho)
    t = chi - 1.0

    # If h=0 and t<=0, the stable fixed point is c*=1 (so m*=0)
    if abs(h) < 1e-15 and t <= 0.0:
        return 1.0

    def g_m(m):
        c = 1.0 - m
        return F_relu(c, chi, rho) - c

    # Asymptotic seed from the kinked equation-of-state:
    #   h + t m - chi*κ m^{3/2} ≈ 0
    m0_h = (h / (max(chi, 1e-12) * KAPPA)) ** (2.0 / 3.0) if h > 0 else 0.0
    m0_t = (t / (max(chi, 1e-12) * KAPPA)) ** 2 if t > 0 else 0.0
    m0 = max(m0_h, m0_t, 1e-8)
    m0 = float(np.clip(m0, 1e-8, 1.0))

    m_lo = max(1e-12, m0 / 100.0)
    m_hi = min(1.9, m0 * 100.0)
    f_lo = g_m(m_lo)
    f_hi = g_m(m_hi)

    for _ in range(80):
        if f_lo * f_hi < 0:
            break
        m_lo = max(1e-12, m_lo / 2.0)
        m_hi = min(1.9, m_hi * 2.0)
        f_lo = g_m(m_lo)
        f_hi = g_m(m_hi)
        if m_lo <= 1e-12 and m_hi >= 1.9:
            break

    if f_lo * f_hi > 0:
        # fallback scan
        ms = np.logspace(-12, 0, 800)
        fs = np.array([g_m(m) for m in ms])
        idx = np.where(fs[:-1] * fs[1:] < 0)[0]
        if len(idx) == 0:
            # iteration fallback
            c = 0.0
            for _ in range(500000):
                c_new = F_relu(c, chi, rho)
                c_new = float(np.clip(c_new, -0.999999999999, 0.999999999999))
                if abs(c_new - c) < 1e-14:
                    return c_new
                c = 0.5 * c + 0.5 * c_new
            return c
        a, b = ms[idx[0]], ms[idx[0] + 1]
        m_star = brentq(g_m, a, b, xtol=1e-14, rtol=1e-12, maxiter=500)
    else:
        m_star = brentq(g_m, m_lo, m_hi, xtol=1e-14, rtol=1e-12, maxiter=500)

    return 1.0 - float(m_star)


# ReLU kink coefficient in expansion near c=1: F_base(1-m)=1-m + kappa m^{3/2} + ...
KAPPA = 2 * np.sqrt(2) / (3 * np.pi)
print(f"ReLU kink coefficient κ = {KAPPA:.8f}")


In [ ]:
def loglog_fit(x, y, fit_slice=slice(None)):
    """
    Fit log10(y) = a log10(x) + b on x,y>0.
    Returns (a, b, se_a, se_b) where se_a, se_b are standard errors.
    """
    x = np.asarray(x)[fit_slice]
    y = np.asarray(y)[fit_slice]
    m = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    lx = np.log10(x[m])
    ly = np.log10(y[m])
    # Use cov=True to get covariance matrix
    coeffs, cov = np.polyfit(lx, ly, 1, cov=True)
    a, b = coeffs
    se_a = np.sqrt(cov[0, 0])
    se_b = np.sqrt(cov[1, 1])
    return float(a), float(b), float(se_a), float(se_b)


def line_from_logfit(x, a, b):
    """Return y = 10^b * x^a."""
    return (10.0**b) * (np.asarray(x) ** a)


## Critical exponents without dropout

We measure three scaling laws:

1. $\xi \sim \epsilon^{-\nu}$ as $\chi\to 1^-$ (ordered side), where $\epsilon=1-\chi$ and $\xi = -1/\ln\chi$.
2. $m_* \sim (\chi-1)^\beta$ as $\chi\to 1^+$ (chaotic side), where $m_* = 1-c_*$.
3. At $\chi=1$, the approach to perfect alignment is algebraic:
   \[
   m_\ell = 1-c_\ell \sim \ell^{-p}.
   \]


In [ ]:
# --- Settings ---
sigma_b2_tanh = 0.1  # tanh bias variance (fixed)
rho_no = 1.0  # no dropout


# Find tanh critical sigma_w^2 by solving chi = 1 at rho = 1
def tanh_chi_minus1(sigw2):
    _, chi, _, _ = tanh_chi_g_h(sigw2, sigma_b2_tanh, rho_no)
    return chi - 1.0


sigw2_c = brentq(tanh_chi_minus1, 0.1, 10.0, xtol=1e-14, rtol=1e-12)
q_c, chi_c, g_c, h_c = tanh_chi_g_h(sigw2_c, sigma_b2_tanh, rho_no)
print(f"tanh critical sigma_w^2 = {sigw2_c:.8f}, chi={chi_c:.8f}, q*={q_c:.6f}")

# ----------------------------
# (a) correlation length exponent nu, using t = chi - 1
# ----------------------------
# Approach criticality from below: chi -> 1^- corresponds to t -> 0^-
tvals_below = -np.logspace(-2, -1, 20)  # t < 0
chi_vals = 1.0 + tvals_below  # chi = 1 + t

xi = -1.0 / np.log(chi_vals)

# Fit xi ~ |t|^{-nu}
a_xi, b_xi, se_a_xi, se_b_xi = loglog_fit(np.abs(tvals_below), xi)
nu = -a_xi
se_nu = se_a_xi

xi_fit = line_from_logfit(np.abs(tvals_below), a_xi, b_xi)

# ----------------------------
# (b) order parameter exponent beta: m* vs t = chi - 1
# ----------------------------
# Smooth tanh: vary sigma_w^2 above critical so chi > 1, hence t > 0
sigw2_grid = sigw2_c * (1.0 + np.logspace(-2, -1, 20))
chi_t = []
m_t = []
for sw2 in sigw2_grid:
    q, chi, _, _ = tanh_chi_g_h(sw2, sigma_b2_tanh, rho_no)
    if chi <= 1.0:
        continue
    cstar = cstar_tanh(sw2, sigma_b2_tanh, rho_no)
    chi_t.append(chi)
    m_t.append(1.0 - cstar)

chi_t = np.array(chi_t)
m_t = np.array(m_t)
tvals_t = chi_t - 1.0

# Kinked ReLU: use explicit chi parameter (>1), hence t > 0
chi_r = 1.0 + np.logspace(-2, -1, 20)
m_r = np.array([1.0 - cstar_relu(chi, rho_no) for chi in chi_r])
tvals_r = chi_r - 1.0

# Fit in the smallest half of the range to capture asymptotic scaling
fit_slice_t = slice(0, max(6, len(tvals_t) // 2))
fit_slice_r = slice(0, max(6, len(tvals_r) // 2))

a_beta_t, b_beta_t, se_a_beta_t, se_b_beta_t = loglog_fit(
    tvals_t, m_t, fit_slice=fit_slice_t
)
a_beta_r, b_beta_r, se_a_beta_r, se_b_beta_r = loglog_fit(
    tvals_r, m_r, fit_slice=fit_slice_r
)

beta_t = a_beta_t
se_beta_t = se_a_beta_t
beta_r = a_beta_r
se_beta_r = se_a_beta_r

tline = np.logspace(np.log10(min(tvals_t)), np.log10(max(tvals_t)), 200)
m_fit_t = line_from_logfit(tline, a_beta_t, b_beta_t)

tline_r = np.logspace(np.log10(min(tvals_r)), np.log10(max(tvals_r)), 200)
m_fit_r = line_from_logfit(tline_r, a_beta_r, b_beta_r)

# ----------------------------
# (c) critical relaxation exponent p: m_l ~ l^{-p} at t = 0 (chi = 1)
# ----------------------------
Lmax = 20000


def Fcrit_tanh(c):
    return F_tanh(c, sigw2_c, sigma_b2_tanh, rho_no, qstar=q_c)


# iterate tanh at criticality
c = 0.0
m_l_t = np.empty(Lmax)
for ell in range(Lmax):
    c = Fcrit_tanh(c)
    m_l_t[ell] = 1.0 - c

# iterate ReLU at criticality (chi = 1, rho = 1)
c = 0.0
m_l_r = np.empty(Lmax)
for ell in range(Lmax):
    c = F_relu_base(c)
    m_l_r[ell] = 1.0 - c

ells = np.arange(1, Lmax + 1)

fit_win = (ells > 500) & (ells < 8000)
a_p_t, b_p_t, se_a_p_t, se_b_p_t = loglog_fit(ells[fit_win], m_l_t[fit_win])
a_p_r, b_p_r, se_a_p_r, se_b_p_r = loglog_fit(ells[fit_win], m_l_r[fit_win])

p_t = -a_p_t
se_p_t = se_a_p_t
p_r = -a_p_r
se_p_r = se_a_p_r

mfit_t = line_from_logfit(ells, a_p_t, b_p_t)
mfit_r = line_from_logfit(ells, a_p_r, b_p_r)

# Print summary of Part I exponents with standard errors
print("\n" + "=" * 60)
print("Part I: Critical exponents (no dropout)")
print("=" * 60)
print(f"  ν (correlation length):  {nu:.4f} ± {se_nu:.4f}")
print(f"  β (tanh order param):    {beta_t:.4f} ± {se_beta_t:.4f}")
print(f"  β (ReLU order param):    {beta_r:.4f} ± {se_beta_r:.4f}")
print(f"  p (tanh relaxation):     {p_t:.4f} ± {se_p_t:.4f}")
print(f"  p (ReLU relaxation):     {p_r:.4f} ± {se_p_r:.4f}")


## Dropout scaling

We tune to the **correlation edge** ($\chi=1$) for each keep probability $\rho$, then measure:

- $m_* \sim h^{1/\delta}$ at $\chi=1$
- $\xi \sim h^{-\nu_\rho}$ at $\chi=1$

We also extend $h$ up to $10^{-1}$ so you can clearly see when the small-$h$ scaling starts to bend.


In [ ]:
# Helper for tanh. Tune sigma_w2 so that chi equals target_chi at given rho
def tune_sigw2_for_chi_tanh(target_chi: float, sigma_b2: float, rho: float):
    def f(sw2):
        _, chi, _, _ = tanh_chi_g_h(sw2, sigma_b2, rho)
        return chi - target_chi

    return brentq(f, 0.05, 20.0, xtol=1e-14, rtol=1e-12)


# Choose dropout probability p = 1 - rho in [1e-3, 0.5], so rho in [0.5, 0.999]
p_list = np.concatenate(
    [
        np.logspace(-3, -2, 8),
        np.logspace(-2, np.log10(0.2), 15),
    ]
)
rho_list = 1.0 - p_list
rho_list = np.clip(rho_list, 1e-6, 0.999999999999)

# Smooth tanh at chi = 1
m_s, xi_s, h_s = [], [], []
for rho in rho_list:
    sw2 = tune_sigw2_for_chi_tanh(1.0, sigma_b2_tanh, rho)
    q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2_tanh, rho)
    cstar = cstar_tanh(sw2, sigma_b2_tanh, rho)
    m = 1.0 - cstar
    lam = Fp_tanh(cstar, sw2, sigma_b2_tanh, rho, qstar=q)
    xi_val = -1.0 / np.log(lam)
    m_s.append(m)
    xi_s.append(xi_val)
    h_s.append(h)

m_s = np.array(m_s)
xi_s = np.array(xi_s)
h_s = np.array(h_s)

# Kinked ReLU at chi = 1
m_k, xi_k, h_k = [], [], []
for rho in rho_list:
    chi = 1.0
    h = 1.0 - F_relu(1.0, chi, rho)  # equals (1 - rho) / rho
    cstar = cstar_relu(chi, rho)
    m = 1.0 - cstar
    lam = Fp_relu(cstar, chi, rho)
    xi_val = -1.0 / np.log(lam)
    m_k.append(m)
    xi_k.append(xi_val)
    h_k.append(h)

m_k = np.array(m_k)
xi_k = np.array(xi_k)
h_k = np.array(h_k)

# Fit exponents using the smallest h decade
h_fit_max = 10.0 * min(h_s.min(), h_k.min())
fit_s = h_s <= h_fit_max
fit_k = h_k <= h_fit_max

a_ms, b_ms, se_a_ms, se_b_ms = loglog_fit(h_s[fit_s], m_s[fit_s])
a_mk, b_mk, se_a_mk, se_b_mk = loglog_fit(h_k[fit_k], m_k[fit_k])

delta_s = 1.0 / a_ms
se_delta_s = se_a_ms / (a_ms**2)  # error propagation for 1/x
delta_k = 1.0 / a_mk
se_delta_k = se_a_mk / (a_mk**2)

a_xs, b_xs, se_a_xs, se_b_xs = loglog_fit(h_s[fit_s], xi_s[fit_s])
a_xk, b_xk, se_a_xk, se_b_xk = loglog_fit(h_k[fit_k], xi_k[fit_k])

nu_rho_s = -a_xs
se_nu_rho_s = se_a_xs
nu_rho_k = -a_xk
se_nu_rho_k = se_a_xk

print("\n" + "=" * 60)
print("Part II: Dropout perturbation exponents (small h fits)")
print("=" * 60)
print(f"  Smooth (tanh):")
print(f"    1/δ = {a_ms:.4f} ± {se_a_ms:.4f}  (δ = {delta_s:.3f} ± {se_delta_s:.3f})")
print(f"    ν_ρ = {nu_rho_s:.4f} ± {se_nu_rho_s:.4f}")
print(f"  Kinked (ReLU):")
print(f"    1/δ = {a_mk:.4f} ± {se_a_mk:.4f}  (δ = {delta_k:.3f} ± {se_delta_k:.3f})")
print(f"    ν_ρ = {nu_rho_k:.4f} ± {se_nu_rho_k:.4f}")


### Combined figure: critical exponents and dropout scaling


In [ ]:
# Combine the first two figures into a single multi-panel figure.
fig = plt.figure(figsize=(22, 12))
fig.patch.set_facecolor("white")
gs = fig.add_gridspec(2, 6, height_ratios=[1.0, 1.0], hspace=0.35, wspace=0.35)

# Top row: three panels (from Part I)
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Bottom row: two panels (from Part II)
ax4 = fig.add_subplot(gs[1, 0:3])
ax5 = fig.add_subplot(gs[1, 3:6])

# ----------------------------
# Panel (a): Correlation length xi vs |t| (t < 0)
# ----------------------------
t_abs = np.abs(tvals_below)
ax1.loglog(
    t_abs,
    xi,
    "o",
    color=COLOR_SMOOTH,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Data",
    zorder=3,
)
ax1.loglog(
    t_abs,
    xi_fit,
    "--",
    color=COLOR_FIT,
    linewidth=2.5,
    label=rf"$\nu = {nu:.3f} \pm {se_nu:.3f}$",
    zorder=2,
)
ax1.set_xlabel(r"$|t| = |\,\chi - 1\,|$")
ax1.set_ylabel(r"$\xi = -1/\ln(\chi)$")
style_log_axes(ax1, legend_loc="lower left")

# ----------------------------
# Panel (b): Order parameter m* vs t (t > 0)
# ----------------------------
ax2.loglog(
    tvals_t,
    m_t,
    "o",
    color=COLOR_SMOOTH,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Smooth tanh",
    zorder=3,
)
ax2.loglog(
    tvals_r,
    m_r,
    "s",
    color=COLOR_KINK,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Kinked ReLU",
    zorder=3,
)
ax2.loglog(
    tline,
    m_fit_t,
    "--",
    color=COLOR_FIT,
    linewidth=2.5,
    label=rf"$\beta_\mathrm{{tanh}} = {beta_t:.3f} \pm {se_beta_t:.3f}$",
    zorder=2,
)
ax2.loglog(
    tline_r,
    m_fit_r,
    "--",
    color=COLOR_THEORY,
    linewidth=2.5,
    label=rf"$\beta_\mathrm{{ReLU}} = {beta_r:.3f} \pm {se_beta_r:.3f}$",
    zorder=2,
)
ax2.set_xlabel(r"$t = \chi - 1$")
ax2.set_ylabel(r"$m_* = 1 - c_*$")
style_log_axes(ax2, legend_loc="upper left")

# ----------------------------
# Panel (c): Critical relaxation m_l vs layer ell (t = 0)
# ----------------------------
ax3.loglog(
    ells,
    m_l_t,
    color=COLOR_SMOOTH,
    linewidth=1.5,
    alpha=0.7,
    label="Smooth tanh",
    zorder=2,
)
ax3.loglog(
    ells,
    m_l_r,
    color=COLOR_KINK,
    linewidth=1.5,
    alpha=0.7,
    label="Kinked ReLU",
    zorder=2,
)
ax3.loglog(
    ells,
    mfit_t,
    "--",
    color=COLOR_FIT,
    linewidth=2.5,
    label=rf"$p_\mathrm{{tanh}} = {p_t:.3f} \pm {se_p_t:.3f}$",
    zorder=3,
)
ax3.loglog(
    ells,
    mfit_r,
    "--",
    color=COLOR_THEORY,
    linewidth=2.5,
    label=rf"$p_\mathrm{{ReLU}} = {p_r:.3f} \pm {se_p_r:.3f}$",
    zorder=3,
)
ax3.set_xlabel(r"Layer $\ell$")
ax3.set_ylabel(r"$m_\ell = 1 - c_\ell$")
style_log_axes(ax3, legend_loc="upper right")

# ----------------------------
# Panel (d): m* vs dropout field h at chi=1
# ----------------------------
ax4.loglog(
    h_s,
    m_s,
    "o",
    color=COLOR_SMOOTH,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Smooth tanh",
    zorder=3,
)
ax4.loglog(
    h_k,
    m_k,
    "s",
    color=COLOR_KINK,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Kinked ReLU",
    zorder=3,
)

hline = np.logspace(
    np.log10(min(h_k.min(), h_s.min())), np.log10(max(h_k.max(), h_s.max())), 200
)
ax4.loglog(
    hline,
    line_from_logfit(hline, a_ms, b_ms),
    "--",
    color=COLOR_FIT,
    linewidth=2.5,
    label=rf"$1/\delta_\mathrm{{tanh}} = {a_ms:.3f} \pm {se_a_ms:.3f}$",
    zorder=2,
)
ax4.loglog(
    hline,
    line_from_logfit(hline, a_mk, b_mk),
    "--",
    color=COLOR_THEORY,
    linewidth=2.5,
    label=rf"$1/\delta_\mathrm{{ReLU}} = {a_mk:.3f} \pm {se_a_mk:.3f}$",
    zorder=2,
)
ax4.set_xlabel(r"Dropout field $h = 1-\bar{F}_\rho(1)$")
ax4.set_ylabel(r"$m_* = 1-c_*$")
style_log_axes(ax4, legend_loc="upper left")

# ----------------------------
# Panel (e): xi vs dropout field h at chi=1
# ----------------------------
ax5.loglog(
    h_s,
    xi_s,
    "o",
    color=COLOR_SMOOTH,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Smooth tanh",
    zorder=3,
)
ax5.loglog(
    h_k,
    xi_k,
    "s",
    color=COLOR_KINK,
    markersize=6,
    markeredgewidth=0.5,
    markeredgecolor="white",
    label="Kinked ReLU",
    zorder=3,
)
ax5.loglog(
    hline,
    line_from_logfit(hline, a_xs, b_xs),
    "--",
    color=COLOR_FIT,
    linewidth=2.5,
    label=rf"$\nu_{{\rho,\mathrm{{tanh}}}} = {-a_xs:.3f} \pm {se_a_xs:.3f}$",
    zorder=2,
)
ax5.loglog(
    hline,
    line_from_logfit(hline, a_xk, b_xk),
    "--",
    color=COLOR_THEORY,
    linewidth=2.5,
    label=rf"$\nu_{{\rho,\mathrm{{ReLU}}}} = {-a_xk:.3f} \pm {se_a_xk:.3f}$",
    zorder=2,
)
ax5.set_xlabel(r"Dropout field $h$")
ax5.set_ylabel(r"$\xi = -1/\ln\lambda,\ \lambda=\bar{F}'(c_*)$")
style_log_axes(ax5, legend_loc="upper right")

plt.savefig(
    RUN_DIR / "combined_exponents_and_dropout_scaling.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()


## Scaling collapse

The fixed points should collapse onto a common curve after rescaling the dropout field and distance from criticality.

### Smooth class (tanh)

The effective equation of state near the edge is
\[
h = -t\,m + \frac{g}{2}m^2,\qquad t\equiv \chi-1,\qquad m\equiv 1-c_*.
\]
This implies the universal collapse
\[
\tilde m \equiv m\sqrt{\frac{g}{2h}},\qquad \tilde t \equiv \frac{t}{\sqrt{2gh}},
\qquad
\tilde m = \sqrt{1+\tilde t^2}+\tilde t.
\]

### Kinked class (ReLU)

Near $c=1$ one has $F(1-m)=1-m+\kappa m^{3/2}+\dots$, so the equation of state becomes
\[
h = -t\,m + \kappa m^{3/2} + \dots
\]
and the universal collapse takes the form
\[
m = \Big(\frac{h}{\kappa}\Big)^{2/3} F(u),\qquad u\equiv \frac{t}{\kappa^{2/3}h^{1/3}}.
\]

The code below sweeps the dropout proxy from $10^{-4}$ to $10^{-1}$.


In [ ]:
# =============================================================================
# Universal scaling functions
# =============================================================================

# Additional colors for multi-curve plots
ZULU_COLORS = [
    "#3A5F3A",  # Original dark military green
    "#C9A961",  # Original gilt/gold
    "#E63946",  # Vivid red
    "#1E88E5",  # Bright blue
    "#7B2CBF",  # Purple
    "#FF6F00",  # Dark orange
    "#00897B",  # Teal
    "#C62828",  # Dark red
    "#5E35B1",  # Deep purple
    "#8B4513",  # Saddle brown
]

h_list = np.logspace(-4, -1, 10)
t_vals = np.linspace(-0.3, 0.3, 50)

# ------------------------------
# Smooth (tanh): full MF data
# ------------------------------
smooth = []
for h_proxy in h_list:
    rho = 1.0 / (1.0 + h_proxy)

    for t in t_vals:
        chi_target = 1.0 + t
        sw2 = tune_sigw2_for_chi_tanh(chi_target, sigma_b2_tanh, rho)

        q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2_tanh, rho)
        cstar = cstar_tanh(sw2, sigma_b2_tanh, rho)
        m = 1.0 - cstar

        smooth.append((h_proxy, t, h, g, m))

smooth = np.array(smooth, dtype=float)
h_proxy_s, t_s, h_s, g_s, m_s = smooth.T

tau_s = -t_s
tau_tilde_s = tau_s / np.sqrt(2.0 * g_s * h_s)
m_tilde_s = m_s * np.sqrt(g_s / (2.0 * h_s))

# ------------------------------
# Kinked (ReLU): analytic MF map
# ------------------------------
kink = []
for h_target in h_list:
    rho = 1.0 / (1.0 + h_target)

    for t in t_vals:
        chi = 1.0 + t
        cstar = cstar_relu(chi, rho)
        m = 1.0 - cstar

        h = 1.0 - F_relu(1.0, chi, rho)

        kink.append((h_target, t, h, m))

kink = np.array(kink, dtype=float)
h_target_k, t_k, h_k, m_k = kink.T

tau_k = -t_k
u_k = tau_k / (KAPPA ** (2.0 / 3.0) * h_k ** (1.0 / 3.0))
m_tilde_k = m_k / ((h_k / KAPPA) ** (2.0 / 3.0))

# ------------------------------
# Universal scaling curves
# ------------------------------
tau_line = np.linspace(-6, 6, 600)
mtilde_s_theory = np.sqrt(1.0 + tau_line**2) - tau_line


def kink_universal(u):
    """Universal kinked scaling function: solve y^3 + u y^2 - 1 = 0, return y^2."""
    u = np.asarray(u, dtype=float)
    out = np.empty_like(u)
    for i, ui in enumerate(u):
        coeff = [1.0, ui, 0.0, -1.0]
        roots = np.roots(coeff)
        real = roots[np.isclose(roots.imag, 0.0, atol=1e-10)].real
        real = real[real > 0]
        out[i] = real.max() ** 2 if len(real) else np.nan
    return out


u_line = np.linspace(-6, 6, 600)
mtilde_k_theory = kink_universal(u_line)

# ------------------------------
# Plot: raw curves + collapses
# ------------------------------
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.patch.set_facecolor("white")

# (a) Smooth raw
ax = axs[0, 0]
for idx, hp in enumerate(h_list):
    mask = np.isclose(h_proxy_s, hp)
    mask0 = mask & np.isclose(t_s, 0.0, atol=1e-12)
    h0 = float(np.median(h_s[mask0])) if np.any(mask0) else float(np.median(h_s[mask]))
    color = ZULU_COLORS[idx]
    ax.plot(
        t_s[mask],
        m_s[mask],
        "o",
        ms=5,
        color=color,
        markeredgewidth=0.5,
        markeredgecolor="white",
        label=rf"$h\approx {h0:.0e}$",
        alpha=0.9,
    )
ax.set_xlabel(r"$t=\chi-1$", fontsize=12)
ax.set_ylabel(r"$m=1-c_*$", fontsize=12)
ax.grid(True, which="major", ls="-", alpha=0.3, linewidth=0.8, color=COLOR_GRID)
ax.grid(True, which="minor", ls=":", alpha=0.15, linewidth=0.5, color=COLOR_GRID)
ax.set_axisbelow(True)
ax.legend(
    frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc="upper left"
)

# (b) Smooth collapse
ax = axs[0, 1]
for idx, hp in enumerate(h_list):
    mask = np.isclose(h_proxy_s, hp)
    mask0 = mask & np.isclose(t_s, 0.0, atol=1e-12)
    h0 = float(np.median(h_s[mask0])) if np.any(mask0) else float(np.median(h_s[mask]))
    color = ZULU_COLORS[idx]
    ax.plot(
        tau_tilde_s[mask],
        m_tilde_s[mask],
        "o",
        ms=5,
        color=color,
        markeredgewidth=0.5,
        markeredgecolor="white",
        label=rf"$h\approx {h0:.0e}$",
        alpha=0.9,
    )
ax.plot(
    tau_line, mtilde_s_theory, "-", color="#2D2D2D", lw=2.5, label="Theory", zorder=10
)
ax.set_xlabel(r"$\tilde{\tau}=(1-\chi)/\sqrt{2gh}$", fontsize=12)
ax.set_ylabel(r"$\tilde{m}=m\sqrt{g/(2h)}$", fontsize=12)
ax.set_xlim(-1, 2)
ax.set_ylim(0, 2)
ax.grid(True, which="major", ls="-", alpha=0.3, linewidth=0.8, color=COLOR_GRID)
ax.grid(True, which="minor", ls=":", alpha=0.15, linewidth=0.5, color=COLOR_GRID)
ax.set_axisbelow(True)
ax.legend(
    frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc="upper right"
)

# (c) Kinked raw
ax = axs[1, 0]
for idx, htar in enumerate(h_list):
    mask = np.isclose(h_target_k, htar)
    color = ZULU_COLORS[idx]
    ax.plot(
        t_k[mask],
        m_k[mask],
        "s",
        ms=5,
        color=color,
        markeredgewidth=0.5,
        markeredgecolor="white",
        label=rf"$h={htar:.0e}$",
        alpha=0.9,
    )
ax.set_xlabel(r"$t=\chi-1$", fontsize=12)
ax.set_ylabel(r"$m=1-c_*$", fontsize=12)
ax.grid(True, which="major", ls="-", alpha=0.3, linewidth=0.8, color=COLOR_GRID)
ax.grid(True, which="minor", ls=":", alpha=0.15, linewidth=0.5, color=COLOR_GRID)
ax.set_axisbelow(True)
ax.legend(
    frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc="upper left"
)

# (d) Kinked collapse
ax = axs[1, 1]
for idx, htar in enumerate(h_list):
    mask = np.isclose(h_target_k, htar)
    color = ZULU_COLORS[idx]
    ax.plot(
        u_k[mask],
        m_tilde_k[mask],
        "s",
        ms=5,
        color=color,
        markeredgewidth=0.5,
        markeredgecolor="white",
        label=rf"$h={htar:.0e}$",
        alpha=0.9,
    )
ax.plot(
    u_line, mtilde_k_theory, "-", color="#2D2D2D", lw=2.5, label="Theory", zorder=10
)
ax.set_xlabel(r"$u=(1-\chi)/(\kappa^{2/3}h^{1/3})$", fontsize=12)
ax.set_ylabel(r"$m/(h/\kappa)^{2/3}$", fontsize=12)
ax.set_xlim(-1.25, 2)
ax.set_ylim(0, 2)
ax.grid(True, which="major", ls="-", alpha=0.3, linewidth=0.8, color=COLOR_GRID)
ax.grid(True, which="minor", ls=":", alpha=0.15, linewidth=0.5, color=COLOR_GRID)
ax.set_axisbelow(True)
ax.legend(
    frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc="upper right"
)

plt.tight_layout(pad=1.5)
plt.savefig(
    RUN_DIR / "scaling_collapse.png", dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
